# Code mẫu — Feature extraction

Feature extraction giữ nguyên encoder, rồi dùng các vector nó tạo ra cho một việc mới. Ba ví dụ dưới đây kiểm lần lượt ba bước: gộp các vector token thành vector câu, đo độ gần giữa các vector, và học một lớp đầu ra mới trên biểu diễn có sẵn.

## Ví dụ 1: Gộp token mà không lẫn padding vào câu

Một encoder giả lập trả về ba vector cho mỗi câu, mỗi vector có hai tọa độ. Câu đầu chỉ có hai token thật là `(2, 0)` và `(0, 4)`, nên vector câu đúng phải là `(1, 2)`. Ô padding được cố tình đặt giá trị rất lớn, `(90, 90)`, để nếu nó lọt vào phép tính thì ta thấy ngay.

In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)
hidden = torch.tensor([[[2., 0.], [0., 4.], [90., 90.]],
                       [[1., 1.], [3., 1.], [2., 4.]]])
mask = torch.tensor([[1, 1, 0], [1, 1, 1]], dtype=torch.bool)

def masked_mean(values, valid):
    counts = valid.sum(dim=1, keepdim=True)
    if (counts == 0).any():
        raise ValueError("cau khong co token hop le")
    weights = valid.unsqueeze(-1).to(values.dtype)
    return (values * weights).sum(dim=1) / counts

pooled = masked_mean(hidden, mask)
independent = np.stack([row[keep].mean(axis=0)
                        for row, keep in zip(hidden.numpy(), mask.numpy())])
assert np.allclose(pooled.numpy(), independent)
print("vector cau:", pooled.tolist())
print("khong mask, cau dau:", [round(float(v), 3) for v in hidden.mean(dim=1)[0]])
print("shape sau pooling:", tuple(pooled.shape))
try:
    masked_mean(hidden[:1], torch.zeros((1, 3), dtype=torch.bool))
except ValueError as error:
    print("cau rong:", str(error))

vector cau: [[1.0, 2.0], [2.0, 2.0]]
khong mask, cau dau: [30.667, 31.333]
shape sau pooling: (2, 2)
cau rong: cau khong co token hop le


Có mask, câu đầu cho đúng `[1, 2]`, vì padding bị loại khỏi cả tổng lẫn số đếm. Ta còn kiểm lại bằng một cách tính khác, lọc từng câu bằng NumPy, và hai cách khớp nhau. Lấy trung bình thường trên cả ba ô thì vector bị kéo lên `[30.667, 31.333]`. Shape vẫn đúng `(2, 2)`, nên lỗi này không tự lộ ra. Một câu toàn padding thì bị từ chối, vì không có token nào để lấy trung bình.

## Ví dụ 2: Cosine xét hướng, tích vô hướng xét cả độ dài

Một truy vấn có vector `(1, 0)`. Kho có ba vector: A là `(2, 0)`, B là `(9, 3)`, C là `(0, 2)`. Các tọa độ này tự đặt để kiểm phép đo, không lấy từ model thật nào. A cùng hướng hoàn toàn với truy vấn. B lệch hướng một chút nhưng dài hơn nhiều.

In [ ]:
import numpy as np

query = np.array([[1., 0.]])
database = np.array([[2., 0.], [9., 3.], [0., 2.]])
ids = np.array(["A", "B", "C"])

def unit_rows(values):
    lengths = np.linalg.norm(values, axis=1, keepdims=True)
    if np.any(lengths == 0):
        raise ValueError("vector 0 khong co huong")
    return values / lengths

dot = (query @ database.T)[0]
cosine = (unit_rows(query) @ unit_rows(database).T)[0]
reference_B = 9 / np.sqrt(9**2 + 3**2)
assert np.isclose(cosine[1], reference_B)
normalized_distance = np.linalg.norm(unit_rows(database) - unit_rows(query), axis=1)
assert np.allclose(normalized_distance**2, 2 - 2 * cosine)
print("dot:", dot.tolist(), "chon:", ids[dot.argmax()])
print("cosine:", cosine.round(4).tolist(), "chon:", ids[cosine.argmax()])
print("khoang cach sau normalize:", normalized_distance.round(4).tolist())
try:
    unit_rows(np.array([[0., 0.]]))
except ValueError as error:
    print("vector 0:", str(error))

dot: [2.0, 9.0, 0.0] chon: B
cosine: [1.0, 0.9487, 0.0] chon: A
khoang cach sau normalize: [0.0, 0.3204, 1.4142]
vector 0: vector 0 khong co huong


Tích vô hướng chọn B, vì 9 lớn hơn 2. Cosine chọn A, vì A cùng hướng tuyệt đối với truy vấn, cosine bằng 1. Khi đã đưa mọi vector về độ dài 1, khoảng cách Euclid và cosine luôn cho cùng thứ hạng. Dòng assert kiểm đúng hệ thức $d^2 = 2 - 2\cos$.

Chọn cách đo nào là một quyết định. Độ dài vector có mang thông tin cho bài của ta hay không? Nếu không, dùng cosine. Cosine 0.9487 của B chỉ cho biết B gần truy vấn về hướng, chưa phải xác suất B cùng nhãn với truy vấn. Vector 0 không có hướng, nên bị từ chối.

## Ví dụ 3: Encoder học trước, lớp đầu ra mới học nhãn mới

Dữ liệu nguồn là 64 điểm trên một lưới hai chiều, gán bốn nhãn theo góc phần tư. Ta train một encoder `2 → 8 → Tanh`, kèm một lớp đầu ra bốn lớp, trên bài nguồn này. Bài mới là phân loại điểm nằm phía nào của đường `x + y = 0`. Với bài mới, encoder được giữ nguyên, chỉ có một lớp đầu ra mới được học. Mọi điểm ở đây đều là dữ liệu tổng hợp.

In [ ]:
import copy
import numpy as np
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)
axis = torch.tensor([-1., -.7, -.4, -.1, .1, .4, .7, 1.])
source_X = torch.cartesian_prod(axis, axis)
source_y = (2 * (source_X[:, 0] > 0) + (source_X[:, 1] > 0)).long()
encoder = nn.Sequential(nn.Linear(2, 8), nn.Tanh())
source_head = nn.Linear(8, 4)
source_model = nn.Sequential(encoder, source_head)
source_model.train()
optimizer = torch.optim.Adam(source_model.parameters(), lr=0.04)
for _ in range(100):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(source_model(source_X), source_y)
    loss.backward()
    optimizer.step()

generator = torch.Generator().manual_seed(420)
target_X = torch.rand((72, 2), generator=generator) * 2 - 1
target_y = (target_X.sum(dim=1) > 0).long()
train_X, valid_X = target_X[:40], target_X[40:]
train_y, valid_y = target_y[:40], target_y[40:]
encoder.eval()
source_model.eval()
encoder.requires_grad_(False)
encoder.zero_grad(set_to_none=True)
before = copy.deepcopy(encoder.state_dict())
# no_grad cho ra tensor thường. Lớp đầu ra ở bước sau vẫn được tính gradient bình thường.
with torch.no_grad():
    train_Z, valid_Z = encoder(train_X), encoder(valid_X)
    source_correct = int((source_model(source_X).argmax(1) == source_y).sum())
print("nguon dung:", source_correct, "/", len(source_X))
print("embedding train/valid:", tuple(train_Z.shape), tuple(valid_Z.shape))

nguon dung: 64 / 64
embedding train/valid: (40, 8) (32, 8)


Encoder học đúng hết 64 điểm nguồn. Giờ mỗi điểm của bài mới được biến thành một vector tám chiều, 40 điểm cho train và 32 điểm cho validation. Lớp đầu ra bốn lớp của bài nguồn bị bỏ đi, vì nó trả lời câu hỏi khác.

Ta khóa encoder bằng `requires_grad_(False)` và lưu lại một bản trọng số để lát nữa so. Các vector được tính một lần trong `no_grad()`, rồi giữ lại để dùng nhiều lần. Tính trong `no_grad()` chỉ có nghĩa là không ghi graph của encoder. Lớp đầu ra mới vẫn học bình thường trên các vector này.

In [ ]:
head = nn.Linear(8, 2)
optimizer = torch.optim.Adam(head.parameters(), lr=0.04)
for _ in range(100):
    head.train()
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(head(train_Z), train_y)
    loss.backward()
    optimizer.step()
head.eval()
with torch.inference_mode():
    prediction = head(valid_Z).argmax(1)
correct = int((prediction == valid_y).sum())
encoder_unchanged = all(torch.equal(before[k], value)
                        for k, value in encoder.state_dict().items())
assert encoder_unchanged and all(p.grad is None for p in encoder.parameters())
assert np.array_equal(target_y.numpy(), (target_X.numpy().sum(axis=1) > 0).astype(int))
print("validation dung:", correct, "/", len(valid_y))
print("encoder giu nguyen:", encoder_unchanged)
print("tham so head:", sum(p.numel() for p in head.parameters()))

validation dung: 31 / 32
encoder giu nguyen: True
tham so head: 18


Lớp đầu ra mới đoán đúng 31 trên 32 điểm validation, là những điểm nó chưa thấy lúc học. Dòng “encoder giữ nguyên” xác nhận trọng số encoder không hề đổi, và encoder cũng không nhận gradient nào. Lớp đầu ra chỉ có $8\cdot2+2=18$ tham số, nên học rất nhanh.

Các vector đã tính sẵn chỉ dùng lại được khi encoder và cách tiền xử lý không đổi. Bài sau sẽ fine-tune, tức cho cả encoder học tiếp. Khi đó encoder thay đổi sau mỗi bước, nên phải tính lại vector, không dùng bản đã lưu.